# Sesión 6 · RAG intermedio: chunking y recuperación

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos de la sesión

Al terminar este notebook podrás:

1. **Ingerir documentos reales** (Markdown, HTML, DOCX y PDF), conservando su estructura, y explicar cuándo hace falta **OCR**.
2. **Comparar estrategias de fragmentado** (*chunking*) **con números**: tamaño fijo, oraciones, recursivo, semántico, por sección y padre-hijo.
3. **Combinar búsqueda léxica (BM25) y semántica** con fusión **RRF** (búsqueda híbrida), y saber cuándo cada una falla.
4. **Filtrar por metadatos**, **reordenar candidatos (reranking)** y **reescribir la consulta** cuando la pregunta es parte de una conversación.
5. **Evaluar la recuperación** con *hit@k* y *MRR*, y leer los resultados con criterio (ruido, costo de contexto).

> ⚠️ **Sigue sin ser un agente.** El flujo continúa siendo fijo: nadie decide qué herramienta usar. Lo que mejora es **la calidad de lo que llega al prompt**.

## 🗺️ Mapa del notebook

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| 1 | De un RAG simple a uno intermedio: dónde falla y cómo medirlo | No (embeddings locales) |
| 2 | Ingesta multiformato: HTML, DOCX, PDF y OCR | No |
| 3 | Estrategias de chunking | No (embeddings locales) |
| 4 | **Laboratorio de chunking** (con padre-hijo) | No (embeddings locales) |
| 5 | Búsqueda léxica: BM25 | No |
| 6 | Búsqueda híbrida y fusión RRF | No |
| 7 | Metadatos y filtros | No |
| 8 | Reranking | **Sí, centavos** (o gratis con modelo local) |
| 9 | Reescritura de la consulta y multi-consulta | **Sí, centavos** (o gratis con modelo local) |
| 10 | El pipeline completo y la práctica | **Sí, centavos** (opcional) |
| 11 | Cierre | No |

> 💰 Las celdas que llaman al LLM llevan **💰 FYI** y una estimación. Con Anthropic u OpenAI, toda la sesión cuesta **menos de US$ 0.02**; con un modelo local en LM Studio, cuesta 0.

---
# Parte 0 · Preparar el entorno

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s06-rag-intermedio y selecciona el kernel .venv (ver README)."
)

In [ ]:
import json
import os
import time
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

from cliente_llm import Gasto, hay_api_key, local_disponible, modelo_local, variable_api_key
from embeddings import embeddings_disponibles, modelo_embeddings, proveedor_embeddings
from fragmentado import ESTRATEGIAS, fragmentar_corpus, fragmentar_doc, fragmentar_padre_hijo, resumen_fragmentos
from ingesta import leer_carpeta, leer_documento_multi
from rag import proveedor_generacion, proveedores_disponibles
from recuperacion import (
    BM25, MODOS, buscar_bm25, buscar_hibrido, buscar_multiconsulta, buscar_vectorial, construir_indice,
    contiene_clave, evaluar_busqueda, expandir_a_padre, expandir_consulta, fusion_rrf, reescribir_consulta,
    rerank_llm, responder_avanzado, tokenizar,
)

HAY_EMB, detalle = embeddings_disponibles()
print(("✅ Embeddings: " if HAY_EMB else "❌ Embeddings: ") + detalle)

### 🔌 Quién ejecuta los pasos con LLM

Tres técnicas de esta sesión (reranking, reescritura de la consulta y multi-consulta) **le piden ayuda a un LLM**, además de la generación de la respuesta. Se elige con `RAG_PROVIDER` (o, si no existe, `LLM_PROVIDER`) en tu `.env`: `anthropic`, `openai` o `local` (LM Studio).

> 🔎 **Una advertencia que verás medida:** un modelo local de 3B parámetros sirve para ver el flujo, pero hace estas tareas peor que un modelo de proveedor. Es parte de la lección: **cada técnica que añade un LLM debe ganarse su lugar con una medición**.

In [ ]:
GEN = proveedor_generacion()
if GEN == "local":
    HAY_LLM, msg = local_disponible(modelo_local())
else:
    HAY_LLM, msg = hay_api_key(GEN), f"falta {variable_api_key(GEN)} en tu .env"
print(f"LLM para los pasos con IA: {GEN} · {'✅ disponible' if HAY_LLM else '❌ ' + str(msg)}")

### 🔌 Interruptor general de gasto

💰 **FYI:** con `False`, ninguna celda llama al LLM (las partes 8, 9 y 10 se omiten). Te recomendamos un límite de gasto mensual en tu cuenta (ver `docs/COSTOS.md`).

In [ ]:
EJECUTAR_LLAMADAS_DE_PAGO = True   # ← ponla en False para no llamar al LLM
PUEDE_GASTAR = EJECUTAR_LLAMADAS_DE_PAGO and HAY_LLM and HAY_EMB
print("Celdas con LLM activas:", PUEDE_GASTAR)
gasto = Gasto()

---
# Parte 1 · De un RAG simple a uno intermedio

## Lo que tenías al terminar la Sesión 5

```
INGESTA     un .md  ─► fragmentar (sección / oraciones / fijo) ─► vectorizar ─► guardar
CONSULTA    pregunta ─► vectorizar ─► los k más parecidos ─► prompt ─► LLM ─► respuesta con citas
```

Funcionaba con un manual **ordenado**, con **una sola** forma de buscar. En la vida real aparecen cuatro problemas:

| Problema | Ejemplo | Técnica que lo ataca (Parte) |
| --- | --- | --- |
| Los documentos **no son Markdown** | Un contrato en Word, una guía en una página web, un boletín en PDF (a veces escaneado) | **Ingesta multiformato** (2) |
| Los fragmentos **cortan o diluyen** las ideas | Una regla partida a la mitad; una sección entera de 200 palabras con 3 temas | **Estrategias de chunking** (3 y 4) |
| La búsqueda por significado **falla con lo exacto** | "¿Qué indica la alarma OPT-1107?": los códigos casi no tienen "significado" | **BM25 + búsqueda híbrida** (5 y 6) |
| La misma pregunta tiene **respuestas distintas según quién pregunta** | "¿En cuánto tiempo reparan una falla?" ≠ para un hogar y para una empresa | **Metadatos y filtros** (7) |
| La pregunta **no se entiende sola** | "¿Y los domingos?" | **Reescritura de la consulta** (9) |
| El fragmento correcto está **en el puesto 5**, pero solo enviamos 3 | | **Reranking** (8) |

## 🏗️ Analogía de infraestructura

Piensa en un **sistema de observabilidad**: una búsqueda por texto libre (como `grep`) encuentra el código exacto `OPT-1107`, pero no sabe que "caída de voltaje" y "voltaje bajo" son lo mismo; una búsqueda semántica sí lo sabe, pero se pierde con identificadores. Un buen sistema usa **las dos**, **filtra** por entorno o por equipo (`env=prod`), y deja que **un revisor** reordene los candidatos antes de actuar. Eso es exactamente un RAG intermedio.

## Qué corpus usaremos

Cuatro documentos **ficticios** de "Andes Fibra", en cuatro formatos, y **30 preguntas de prueba** con la respuesta esperada. Conviene medir **desde ya** para saber si cada mejora sirve.

In [ ]:
DATOS = Path("datos")
documentos = leer_carpeta(DATOS)
print(f"{'documento':<28}{'tipo':<6}{'secciones':>10}{'palabras':>10}")
for d in documentos:
    palabras = sum(len(t.split()) for _, t in d.secciones)
    print(f"{d.fuente:<28}{d.tipo:<6}{len(d.secciones):>10}{palabras:>10}")
    for aviso in d.avisos:
        print(f"    ⚠️ {aviso}")

Fíjate en el aviso de `acta_escaneada.pdf`: **no se pudo leer ningún texto**. Lo resolvemos en la Parte 2.

## Las preguntas de prueba

Cada pregunta trae una **`clave`**: el texto que debe aparecer en un fragmento para que *ese fragmento* contenga la respuesta. Así podemos medir **cualquier** estrategia de fragmentado, aunque cambien los fragmentos (en la Sesión 5 comparábamos secciones; aquí los fragmentos cambian de forma y tamaño).

| Tipo | Qué pone a prueba | Ejemplo |
| --- | --- | --- |
| `semantica` | Paráfrasis: casi no comparte palabras con el documento | "Si me voy antes del año, ¿cuánto me cobran de multa?" |
| `lexica` | Un código, modelo o número **exacto** | "¿Qué indica la alarma OPT-1107?" |
| `tabla` | Datos que estaban en una tabla (DOCX/PDF) | "¿Qué día hacen el mantenimiento en la zona sur?" |
| `filtro` | La respuesta depende del **tipo de cliente** | "¿En cuánto tiempo deben reparar una falla?" |
| `conversacional` | Necesita el **historial** para entenderse | "¿Y los domingos?" |

In [ ]:
preguntas = json.loads(Path("datos/preguntas_recuperacion.json").read_text(encoding="utf-8"))
por_tipo = {}
for p in preguntas:
    por_tipo.setdefault(p["tipo"], []).append(p)
for tipo, lista in por_tipo.items():
    print(f"{tipo:<15}{len(lista):>3} preguntas  ·  ej.: {lista[0]['pregunta']}")

# Las preguntas "base": se pueden hacer sin filtro ni historial. Con ellas comparamos estrategias y buscadores.
BASE = [p for p in preguntas if p["tipo"] in ("semantica", "lexica", "tabla")]
print(f"\nPreguntas base (semantica + lexica + tabla): {len(BASE)}")

## ¿Cómo medimos? `hit@k`, `MRR` y el costo del contexto

| Métrica | Qué responde |
| --- | --- |
| **hit@1** | ¿En cuántas preguntas el **primer** fragmento contiene la respuesta? |
| **hit@3** | ¿Y entre los **tres** primeros? (los que enviaríamos al LLM) |
| **MRR** (*Mean Reciprocal Rank*) | Promedio de `1 / posición` del primer acierto. 1.0 = siempre primero; 0.5 = suele ser el segundo; 0 = nunca aparece. Premia **estar arriba**, no solo estar. |
| **contexto** | Palabras promedio de los k fragmentos: **es lo que pagas en tokens** al armar el prompt. |

> ⚠️ Esta medición solo mira la **recuperación**, no la respuesta del LLM (igual que en la Sesión 5: si el dato correcto no llega al prompt, ningún prompt lo arregla). Y con 22 preguntas, **1 o 2 aciertos de diferencia son ruido**.

Esta es nuestra **línea base**: la recuperación de la Sesión 5 (fragmentos de ≈70 palabras por oraciones, búsqueda semántica) sobre el corpus nuevo.

In [ ]:
def linea(nombre, r):
    """Imprime una fila de resultados."""
    print(f"{nombre:<38} hit@1 {r['hit1']:>2}/{r['n']:<2} · hit@{r['k']} {r['hitk']:>2}/{r['n']:<2} · MRR {r['mrr']:.2f} · contexto ≈ {r['contexto']:.0f} palabras")

if HAY_EMB:
    fragmentos_base = fragmentar_corpus(documentos, estrategia="oraciones", max_palabras=70)
    indice_base = construir_indice(fragmentos_base)        # vectoriza (local, gratis) y arma el índice
    print("Fragmentos:", resumen_fragmentos(fragmentos_base))
    linea("Línea base (oraciones 70, semántica)",
          evaluar_busqueda(BASE, lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k), k=3))

Ese número es **tu punto de partida**. Cada técnica de las partes siguientes tiene que **superarlo** (o justificar su costo) para ganarse un lugar en el pipeline.

---
# Parte 2 · Ingesta multiformato

## El paso más subestimado

Antes de fragmentar y vectorizar hay que **extraer el texto**. Si en esta etapa entra basura (menús de navegación, pies de página, tablas aplanadas, texto de otra columna), **todo lo demás trabaja con basura**: *garbage in, garbage out*.

```
archivo ─► extraer texto + estructura ─► Documento(secciones) ─► fragmentar ─► vectorizar ─► guardar
```

`ingesta.py` convierte cada formato en la **misma estructura** que ya conoces (`Documento` con secciones `(título, texto)`) y añade `tipo` y `avisos`:

| Formato | Herramienta | Qué hay que cuidar |
| --- | --- | --- |
| **Markdown** | El lector de la Sesión 5 | Casi nada: ya trae estructura |
| **HTML** | `BeautifulSoup` | Quitar `<script>`, `<style>`, menús y pie de página; conservar títulos y listas |
| **DOCX** | `python-docx` | Respetar los estilos de título (*Heading 1/2*); las **tablas** hay que convertirlas |
| **PDF** (con texto) | `pypdf` | Una sección por **página**; las líneas se cortan por el ancho de la hoja; las **tablas se aplanan** |
| **PDF escaneado** | **OCR** | No hay texto: es una imagen |

## 2.1 HTML: quedarse con el contenido, descartar el ruido

In [ ]:
crudo = Path("datos/guia_tecnica.html").read_text(encoding="utf-8")
print("HTML crudo:", len(crudo.split()), "palabras, incluidas las etiquetas")
print("Contiene <script>:", "<script>" in crudo, "· <nav>:", "<nav>" in crudo, "· <footer>:", "<footer>" in crudo)

html = leer_documento_multi("datos/guia_tecnica.html")
todo = " ".join(t for _, t in html.secciones)
print("\nDespués de extraer:", len(todo.split()), "palabras")
print("¿Quedó 'Síguenos en redes sociales' (pie de página)?", "Síguenos" in todo)
print("¿Quedó 'gtag' (script)?", "gtag" in todo)
print("\nSecciones:", [t for t, _ in html.secciones])
print("\nUna sección con lista (se conserva la lista, con guiones):\n")
print(dict(html.secciones)["Códigos de error del router"][:420], "…")

## 2.2 DOCX: estilos de título y tablas

Un Word bien hecho usa **estilos** (*Título 1*, *Título 2*): son la estructura que necesitamos. Nuestro lector arma el nombre de la sección con ambos niveles (`Parte A · Clientes residenciales › Compensaciones`). Ese nombre **no es decoración**: de él sale el metadato `publico` que usaremos para **filtrar** (Parte 7).

Las **tablas** no son texto corrido: las convertimos en una línea por fila (`celda | celda | celda`).

In [ ]:
docx = leer_documento_multi("datos/condiciones_servicio.docx")
for titulo, texto in docx.secciones:
    print(f"• {titulo}  ({len(texto.split())} palabras)")
print("\nLa sección con tabla:\n")
print(dict(docx.secciones)["Parte A · Clientes residenciales › Penalización por cancelación anticipada"])

## 2.3 PDF con texto: cuidado con las tablas

`pypdf` entrega el texto en el orden en que lo guardó el PDF. Un PDF **no tiene filas ni columnas**: tiene letras en posiciones. Por eso la tabla de fechas del boletín **se aplana**: la información sigue ahí, pero ya no es una tabla.

In [ ]:
pdf = leer_documento_multi("datos/boletin_mantenimiento.pdf")
print([t for t, _ in pdf.secciones])
pagina1 = pdf.secciones[0][1]
inicio = pagina1.index("Ventanas por zona")
print("\nLa tabla de ventanas, tal como quedó:\n")
print(pagina1[inicio:inicio + 260], "…")

Así, **la fila de la zona sur queda en el mismo renglón que las demás**. Si el fragmento se corta en mala hora, la fecha puede separarse de su zona. Para PDFs con muchas tablas se usan extractores que reconocen la estructura (por ejemplo `pdfplumber`, o servicios de *document AI*), o se pide a un modelo con visión que lea la página.

## 2.4 PDF escaneado: cuando no hay texto

`acta_escaneada.pdf` es **una imagen de un papel**. No tiene capa de texto: `pypdf` no devuelve nada, y nuestro lector lo **avisa** en lugar de ingerir un documento vacío en silencio.

In [ ]:
acta = leer_documento_multi("datos/acta_escaneada.pdf")
print("Secciones extraídas:", len(acta.secciones))
print("Avisos:", acta.avisos)

### ¿Qué es el OCR?

**OCR** (*Optical Character Recognition*, reconocimiento óptico de caracteres) es convertir una **imagen de texto** en **texto**. Es el paso que falta antes de ingerir escaneos, fotos de documentos o PDFs de "imprimir a imagen".

| Opción | Cuándo | Ojo con |
| --- | --- | --- |
| **Tesseract** (código abierto, local) | Texto impreso limpio; privacidad total | Rinde mal con fotos torcidas, manuscritos y tablas |
| **OCR de la nube** (Azure Document Intelligence, Google Document AI, AWS Textract) | Volumen, tablas, formularios | 💰 pago por página y los documentos salen de tu red |
| **Modelo con visión** (un LLM que "ve" la página) | Documentos complejos, mezcla de tabla y texto | 💰 más caro por página y puede equivocarse con cifras: **verifica** |

> **No lo ejecutamos en esta sesión**: instalar un motor de OCR varía mucho por sistema operativo. Lo importante es **detectarlo** (lo que hace `ingesta.py`) y **decidir** qué hacer: un escaneo ingerido sin OCR **no existe** para tu RAG, y nadie te avisará de que le falta información.

**Reto:** ¿qué pasaría con tu RAG si el 20 % de tus PDFs fueran escaneos y nadie lo notara? Escribe cómo lo detectarías (pista: cuenta las páginas sin texto en cada ingesta).

---
# Parte 3 · Estrategias de chunking

## ¿Por qué importa tanto?

El fragmento es la **unidad de búsqueda** y la **unidad de contexto**: es lo que se vectoriza (y se compara con la pregunta) y lo que se copia al prompt (y cuesta tokens). Por eso tiene dos presiones opuestas:

| Si es… | Al **buscar** | Al **responder** |
| --- | --- | --- |
| **Pequeño** | Vector preciso (un solo tema) | Poco contexto: la respuesta puede quedar partida entre dos fragmentos |
| **Grande** | Vector "diluido" (mezcla varios temas) | Contexto completo, pero **más tokens (dinero)** y más ruido |

No hay un tamaño correcto universal: **se mide**.

## Las estrategias

| Estrategia | Cómo parte | Idea / cuándo |
| --- | --- | --- |
| `fijo` | Ventanas de N palabras, sin mirar la gramática | La más simple y ciega: **corta ideas por la mitad** |
| `oraciones` | Junta oraciones hasta N palabras y repite las últimas (solape) | Buen punto de partida para texto corrido |
| `seccion` | Un fragmento por sección (o por página en un PDF) | Respeta la estructura; las secciones largas **diluyen** |
| `recursivo` | Corta por el separador **más grande que quepa** (párrafo → línea → oración → palabra) y junta vecinos hasta N palabras | La más usada en la práctica (es la idea del *RecursiveCharacterTextSplitter* de LangChain) |
| `semantico` | Mide qué tan parecidas son las **oraciones vecinas** (embeddings) y corta donde el tema cambia | Fragmentos coherentes; **cuesta una vectorización por oración** |
| `padre-hijo` | Se **busca** con hijos pequeños y se **entrega** el padre (la sección) | Precisión al buscar + contexto al responder |

**Metadatos:** cada fragmento guarda `fuente`, `seccion`, `n`, `tipo` y `publico`. Sin ellos no puedes **citar** ni **filtrar**.

## 3.1 Una misma sección, cuatro maneras de partirla

Veamos la sección **"3. Facturación y pagos"** del manual con fragmentos de ≈40 palabras. Mira **dónde termina cada fragmento**.

In [ ]:
manual = next(d for d in documentos if d.fuente == "manual_soporte.md")
facturacion = type(manual)(fuente=manual.fuente, secciones=[s for s in manual.secciones if s[0].startswith("3.")], tipo="md")

def mostrar(estrategia, tam=40):
    frs = fragmentar_doc(facturacion, estrategia=estrategia, max_palabras=tam)
    print(f"━━ {estrategia} ({tam} palabras) → {len(frs)} fragmentos")
    for f in frs:
        p = f.texto.split()
        print(f"   [{f.metadatos['n']}] ({len(p):>2} pal.)  {' '.join(p[:5])} … {' '.join(p[-5:])}")
    print()

for estrategia in ("fijo", "oraciones", "recursivo"):
    mostrar(estrategia)

**Observa:** con `fijo`, los fragmentos **terminan a mitad de frase** (por ejemplo, un número separado de su unidad). Con `oraciones` y `recursivo`, terminan en un punto. Esa diferencia, que parece cosmética, hace que un fragmento sea **leíble** y su vector **coherente**.

## 3.2 Fragmentado semántico: cortar donde cambia el tema

El fragmentado semántico vectoriza **cada oración**, calcula la similitud coseno entre oraciones consecutivas y **corta donde cae más** (las uniones más "flojas"). Así se ve en la sección de *Fallas y soporte técnico*:

In [ ]:
import numpy as np
from embeddings import vectorizar
from rag import _oraciones

fallas = next(t for titulo, t in manual.secciones if titulo.startswith("5."))
oraciones = _oraciones(fallas)
v = np.array(vectorizar(oraciones, tipo="documento"), dtype=np.float32)
v /= np.linalg.norm(v, axis=1, keepdims=True)
similitud = (v[:-1] * v[1:]).sum(axis=1)
corte = np.percentile(similitud, 25)
print(f"{len(oraciones)} oraciones · se cortaría donde la similitud con la siguiente es ≤ {corte:.2f}\n")
for i, s in enumerate(similitud):
    marca = "✂️  CORTE" if s <= corte else ""
    print(f"  {s:.2f}  {oraciones[i][:62]:<62}… {marca}")

Las uniones marcadas son los **cambios de tema** que detectó el modelo. Es elegante, pero **no siempre mejora los resultados** (lo mediremos en la Parte 4) y cuesta más tiempo de cómputo en la ingesta: vectoriza cada oración, no solo cada fragmento.

## 3.3 ¿Cuántos fragmentos salen de cada una?

In [ ]:
print(f"{'estrategia':<11}{'palabras':>9}{'fragmentos':>12}{'mínimo':>8}{'media':>8}{'máximo':>8}")
for estrategia in ESTRATEGIAS:
    for tam in ([70] if estrategia == "seccion" else (40, 70, 120)):
        r = resumen_fragmentos(fragmentar_corpus(documentos, estrategia=estrategia, max_palabras=tam))
        print(f"{estrategia:<11}{('—' if estrategia == 'seccion' else tam):>9}{r['n']:>12}{r['min']:>8}{r['media']:>8}{r['max']:>8}")

`seccion` produce **pocos fragmentos grandes** (hasta ≈200 palabras): es lo que pagarás en el prompt. `semantico` produce **muchos fragmentos de tamaño desigual**. En la siguiente parte vemos qué le pasa a la calidad.

---
# Parte 4 · Laboratorio de chunking

Probamos **todas las combinaciones** de estrategia y tamaño con la **misma búsqueda** (semántica) y las mismas 22 preguntas base. Para cada una: fragmentar → vectorizar (con caché) → medir.

> ⏱️ La primera vez tarda ≈ 20 a 40 segundos con el modelo local (vectoriza todo el corpus varias veces). Las siguientes veces la caché lo hace casi instantáneo.

In [ ]:
def probar(estrategia, tam, modo="vectorial", preguntas=BASE, k=3):
    """Fragmenta el corpus con una configuración, arma el índice y mide."""
    frs = fragmentar_corpus(documentos, estrategia=estrategia, max_palabras=tam)
    ix = construir_indice(frs)
    r = evaluar_busqueda(preguntas, lambda p, k: MODOS[modo](ix, p["pregunta"], k, p.get("filtro")), k)
    r["fragmentos"] = len(frs)
    return r

if HAY_EMB:
    t0 = time.perf_counter()
    resultados = []
    for estrategia in ESTRATEGIAS:
        for tam in ([70] if estrategia == "seccion" else (40, 70, 120)):
            r = probar(estrategia, tam)
            r.update(estrategia=estrategia, tam="—" if estrategia == "seccion" else tam)
            resultados.append(r)
    resultados.sort(key=lambda r: (-r["mrr"], r["contexto"]))
    print(f"({time.perf_counter() - t0:.0f} s)\n")
    print(f"{'estrategia':<11}{'palabras':>9}{'fragm.':>8}{'hit@1':>8}{'hit@3':>8}{'MRR':>7}{'contexto':>10}")
    for r in resultados:
        print(f"{r['estrategia']:<11}{r['tam']:>9}{r['fragmentos']:>8}{r['hit1']:>5}/{r['n']:<2}{r['hitk']:>5}/{r['n']:<2}{r['mrr']:>7.2f}{r['contexto']:>10.0f}")

## Cómo leer esta tabla (la parte importante)

1. **Mira la columna *contexto* antes de coronar a un ganador.** `seccion` suele quedar arriba en MRR: la sección entera casi siempre **contiene** la respuesta. Pero entrega **2 o 3 veces más palabras** al LLM que los fragmentos de 40 a 70 palabras: más tokens, más dinero y más ruido. Un fragmento grande "gana" en esta métrica **porque es grande**.
2. **`fijo` con 40 palabras** es el peor: sin respetar la gramática y con ventanas pequeñas, corta datos por la mitad.
3. **`oraciones`, `recursivo` y `semantico` quedan muy parejos.** Con 22 preguntas, 1 o 2 aciertos de diferencia son **ruido**: no declares ganador a uno por una pregunta.
4. **No hay ganador universal.** Cambia el modelo de embeddings, los documentos o las preguntas y la tabla cambia. **Lo valioso es el método**: probar, medir y decidir con números (la misma disciplina de la Sesión 3, aplicada a la recuperación).

> 📊 **Resultados de referencia del instructor** (modelo `nomic-embed-text-v2-moe`, búsqueda semántica): `seccion` ≈ 20/22 en hit@1 con ≈ 350 palabras de contexto; `oraciones` de 70 palabras ≈ 18/22 con ≈ 160 palabras; `fijo` de 40 palabras ≈ 13/22. Tus números pueden diferir en 1 o 2 unidades.

## 4.1 El patrón padre-hijo: lo mejor de los dos mundos

Si los fragmentos **pequeños** buscan mejor pero los **grandes** responden mejor, ¿por qué no usar ambos?

```
INGESTA     sección (padre) ─► se parte en hijos de ≈35 palabras ─► se vectorizan los HIJOS
CONSULTA    pregunta ─► se buscan los hijos más parecidos ─► se entrega el PADRE de cada uno (sin repetir)
```

Es como un **índice de libro**: buscas por la entrada precisa, pero lees la página completa.

In [ ]:
if HAY_EMB:
    hijos = [h for d in documentos for h in fragmentar_padre_hijo(d, 35)]
    ix_ph = construir_indice(hijos)
    print("Hijos:", resumen_fragmentos(hijos))
    ej = hijos[40]
    print("\nUn hijo:     ", ej.texto[:100], "…")
    print("Su padre es: ", ej.metadatos["padre_id"], f"({len(ej.metadatos['texto_padre'].split())} palabras)\n")

    linea("Oraciones 70 (línea base)", evaluar_busqueda(BASE, lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k), 3))
    linea("Padre-hijo (hijos de 35)",
          evaluar_busqueda(BASE, lambda p, k: expandir_a_padre(buscar_vectorial(ix_ph, p["pregunta"], 10))[:k], 3))

**Qué verás:** el padre-hijo iguala o supera a la sección entera en hit@1 y MRR, pero **también entrega secciones completas** (contexto grande). Es la técnica correcta cuando tu documento tiene secciones coherentes y quieres **citar la sección completa**; es mala si las secciones son enormes (el LLM recibe mucho que no necesita).

## Ejercicio

1. Cambia el tamaño de los hijos a 25 y a 50. ¿Cambia algo?
2. ¿Qué pasaría con un PDF de una sola sección de 3 000 palabras (un libro sin títulos)? ¿Qué sería el "padre"?
3. Corre la **práctica** (`practica/laboratorio_chunking.py`) con `--tipos lexica` y compara con `--tipos semantica`. ¿La mejor estrategia es la misma?

---
# Parte 5 · Búsqueda léxica: BM25

## El problema de la búsqueda semántica

Un modelo de embeddings **aprende significado**, y lo hace muy bien con paráfrasis. Pero los **identificadores** (`OPT-1107`, `E-217`, `0800-26337`, un número de factura, un SKU) casi no tienen significado: son **etiquetas exactas**. Para el modelo, `OPT-1102` y `OPT-1107` son casi el mismo texto.

## La solución clásica: BM25

**BM25** es el algoritmo de búsqueda por **palabras** que usan Elasticsearch, OpenSearch y Lucene desde hace décadas. Puntúa más a un fragmento cuando:

- contiene las palabras de la consulta,
- esas palabras son **raras** en el corpus (`OPT-1107` pesa mucho; "el", poco): es el **IDF**,
- aparecen varias veces, pero con **rendimientos decrecientes** (parámetro `k1`),
- y **sin favorecer a los fragmentos más largos** (parámetro `b`).

No entiende significado: "contraseña" y "clave" son palabras distintas. A cambio, **nunca se confunde con un código**.

## 5.1 Cómo se "ve" un texto: el tokenizador

In [ ]:
for texto in ["¿Qué significa el error E-217?", "Cancelar el contrato de Negocio", "Llamar al 0800-26337"]:
    print(f"{texto!r}\n   → {tokenizar(texto)}")
print("\nSin recortar palabras (prefijo=None):", tokenizar("Cancelación y cancelar", prefijo=None))
print("Con recorte a 5 letras (por defecto):  ", tokenizar("Cancelación y cancelar"))

Tres decisiones del tokenizador, y por qué:

- **Minúsculas y sin acentos:** "Dólares" = "dolares".
- **Códigos completos:** `e-217` se conserva **entero** (y también sus partes `e` y `217`), para que una búsqueda por el código exacto lo encuentre.
- **Recorte a 5 letras** (un *stemming* ingenuo): junta "cancelar", "cancelación" y "cancela" en una sola raíz. Es tosco, pero funciona sin librerías de lingüística. **No se aplica a tokens con dígitos**, para no estropear códigos.

## 5.2 Cuándo gana BM25 y cuándo pierde

In [ ]:
if HAY_EMB:
    for consulta in ["¿Qué indica la alarma OPT-1107?", "¿Pueden llevarme el servicio a otra casa?"]:
        print(f"━━ {consulta}")
        for nombre, funcion in (("semántica", buscar_vectorial), ("BM25", buscar_bm25)):
            r = funcion(indice_base, consulta, 1)[0]
            print(f"   {nombre:<10} → {r.metadatos['seccion'][:42]:<42} | {r.texto[:70]}…")
        print()

El primero es un **código** (BM25 lo encuentra, la semántica puede dudar); el segundo es una **paráfrasis** (la semántica entiende "otra casa" = "traslado"; BM25 no tiene ninguna palabra en común). Midámoslo por tipo de pregunta.

In [ ]:
def tabla_por_tipo(buscadores, tipos=("semantica", "lexica", "tabla"), k=3):
    """Una fila por buscador, una columna por tipo de pregunta (aciertos en el primer lugar) y el total."""
    print(f"{'buscador':<26}" + "".join(f"{t:>14}" for t in tipos) + f"{'TOTAL hit@1':>14}{'MRR':>7}")
    for nombre, fn in buscadores.items():
        celdas, total_aciertos, total = [], 0, 0
        for t in tipos:
            qs = [p for p in preguntas if p["tipo"] == t]
            r = evaluar_busqueda(qs, fn, k)
            celdas.append(f"{r['hit1']}/{r['n']}")
            total_aciertos += r["hit1"]; total += r["n"]
        todas = evaluar_busqueda([p for p in preguntas if p["tipo"] in tipos], fn, k)
        print(f"{nombre:<26}" + "".join(f"{c:>14}" for c in celdas) + f"{total_aciertos:>10}/{total:<3}{todas['mrr']:>7.2f}")

if HAY_EMB:
    tabla_por_tipo({
        "Semántica (vectores)": lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k),
        "BM25 (palabras)": lambda p, k: buscar_bm25(indice_base, p["pregunta"], k),
    })

**Lectura:** BM25 gana en las preguntas **léxicas** (códigos, alarmas) y pierde en las **semánticas** (paráfrasis). Ninguna es mejor en general: cada una **cubre el punto ciego de la otra**. Eso nos lleva a combinarlas.

> 📊 **Referencia del instructor** (oraciones de 70 palabras): en las preguntas léxicas, semántica 8/11 y BM25 **11/11**; en las semánticas, semántica 8/9 y BM25 6/9.

---
# Parte 6 · Búsqueda híbrida y fusión RRF

## La idea

Hacemos **las dos búsquedas** y **juntamos** los resultados. El problema: ¿cómo se combinan dos listas con puntajes en escalas incomparables? (El coseno va de 0 a 1; BM25, de 0 a 20 o más.) Sumarlos no tiene sentido.

## RRF: *Reciprocal Rank Fusion*

La solución más usada es **ignorar los puntajes y usar solo la posición**:

$$\text{puntaje}(d) = \sum_{\text{listas}} \frac{\text{peso}}{c + \text{posición}(d)} \qquad (c = 60 \text{ por tradición})$$

Un fragmento que aparece **bien ubicado en ambas listas** suma dos aportes y sube al primer lugar. Un fragmento que solo aparece en una, queda atrás de uno que aparece en las dos. Es una **votación**.

### Un ejemplo a mano

In [ ]:
from almacenes import Resultado

def lista(*ids):
    return [Resultado(i, f"texto {i}", {}, 0.0) for i in ids]

semantica = lista("A", "B", "C")       # el orden según la búsqueda por significado
palabras  = lista("C", "A", "D")       # el orden según BM25
for r in fusion_rrf([semantica, palabras], k=4):
    print(f"{r.id}: {r.puntaje:.5f}")
print()
print("A: 1/(60+1) + 1/(60+2) =", round(1 / 61 + 1 / 62, 5), " (primero en una lista, segundo en la otra)")
print("C: 1/(60+3) + 1/(60+1) =", round(1 / 63 + 1 / 61, 5), " (tercero en una, primero en la otra)")
print("B: 1/(60+2)            =", round(1 / 62, 5), " (solo aparece en una lista)")

## 6.1 ¿Funciona? Comparemos cuatro buscadores

Añadimos una variante: **pesos distintos** (`[1, 2]` = BM25 vale el doble que la semántica).

In [ ]:
if HAY_EMB:
    tabla_por_tipo({
        "Semántica": lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k),
        "BM25": lambda p, k: buscar_bm25(indice_base, p["pregunta"], k),
        "Híbrida (pesos 1 : 1)": lambda p, k: buscar_hibrido(indice_base, p["pregunta"], k),
        "Híbrida (semántica 1 : BM25 2)": lambda p, k: buscar_hibrido(indice_base, p["pregunta"], k, pesos=[1, 2]),
    })

**Lo que probablemente verás, y por qué es una lección honesta:**

- La híbrida con pesos **1 : 1 no siempre supera a la mejor de las dos**: en este corpus pequeño la semántica ya es muy buena, y BM25 "estorba" en las paráfrasis, así que el híbrido iguala o queda **entre las dos**.
- Con pesos que **favorecen a BM25** mejora porque rescata los códigos sin perder demasiado en las paráfrasis. Aquí aparece un riesgo: **elegir los pesos con las mismas preguntas con las que evalúas es hacer trampa** (sobreajuste). En serio, separas preguntas de **ajuste** y de **prueba**.
- La híbrida **gana de verdad cuando el corpus crece**: con miles de fragmentos, la semántica empieza a confundir códigos parecidos, y BM25 es el seguro. También es la forma **estándar** en bases como Elasticsearch, OpenSearch, Pinecone (híbrido), Weaviate o PostgreSQL (`tsvector` + pgvector).

> 📊 **Referencia del instructor** (oraciones de 70 palabras, 22 preguntas): semántica 18/22 y MRR 0.87; BM25 19/22 y 0.86; híbrida 1 : 1, 18/22 y 0.86; híbrida con BM25 doble, **20/22 y 0.91**.

> 🏗️ **Analogía:** la semántica es el **compañero senior que entiende lo que quieres decir**; BM25 es el **`grep`**. Quieres a los dos en la guardia, y alguien que decida cuando discrepan (RRF).

---
# Parte 7 · Metadatos y filtros

## Dos preguntas, el mismo texto, respuestas distintas

> **"¿En cuánto tiempo deben reparar una falla?"**
> - A un cliente **residencial**: 48 horas. 
> - A un cliente **Negocio**: 4 horas.

Un buscador sin contexto devuelve los dos fragmentos (se parecen muchísimo) y el LLM puede **mezclar las respuestas**. La solución es **filtrar por metadatos**: no es una cuestión de "significado", es una **regla de negocio**: *"este usuario es un cliente Negocio"*.

El etiquetado ocurre en la **ingesta** (`publico` se infiere del título de la sección del DOCX), y el filtro se aplica en la **búsqueda**, **antes** de que nada llegue al prompt.

In [ ]:
if HAY_EMB:
    from collections import Counter
    print("Fragmentos por audiencia:", dict(Counter(f.metadatos["publico"] for f in fragmentos_base)))
    print("Fragmentos por tipo de archivo:", dict(Counter(f.metadatos["tipo"] for f in fragmentos_base)))

    pregunta = "¿En cuánto tiempo deben reparar una falla?"
    for etiqueta, filtro in (("SIN filtro", None), ("con publico=negocio", {"publico": "negocio"}), ("con publico=residencial", {"publico": "residencial"})):
        print(f"\n━━ {etiqueta}")
        for r in buscar_hibrido(indice_base, pregunta, 2, filtro):
            print(f"   [{r.metadatos['publico']:<11}] {r.metadatos['seccion'][:50]:<50} | {r.texto[:55]}…")

In [ ]:
if HAY_EMB:
    filtradas = [p for p in preguntas if p["tipo"] == "filtro"]
    print(f"{len(filtradas)} preguntas cuya respuesta depende del tipo de cliente (cada una trae su filtro)\n")
    for etiqueta, usar in (("SIN filtro", False), ("CON filtro", True)):
        for nombre, funcion in (("semántica", buscar_vectorial), ("híbrida", buscar_hibrido)):
            r = evaluar_busqueda(filtradas, lambda p, k: funcion(indice_base, p["pregunta"], k, p["filtro"] if usar else None), 3)
            print(f"{etiqueta:<11} {nombre:<10} hit@1 {r['hit1']}/{r['n']} · hit@3 {r['hitk']}/{r['n']} · MRR {r['mrr']:.2f}")

## Los filtros no son solo para mejorar la calidad: son **seguridad**

Si tu base mezcla documentos de **varios clientes, equipos o niveles de confidencialidad**, el filtro por permisos (`tenant`, `area`, `nivel`) es la **única** barrera real. Reglas de oro:

1. **Filtra en la búsqueda, no en el prompt.** Un prompt que diga "no muestres lo de otros clientes" **no es un control de acceso**: el dato ya llegó al modelo.
2. El filtro lo decide **tu código** (a partir de la sesión del usuario), **nunca el modelo** ni el texto de la pregunta.
3. Los metadatos que usas para filtrar se **asignan en la ingesta** y se **auditan**.
4. Las bases reales (Neon con `WHERE`, Pinecone con `filter`) aplican el filtro **dentro** de la búsqueda; el almacén en memoria de esta sesión hace lo mismo.

> En la Sesión 5 lo mencionamos como riesgo (*fuga de permisos*). Aquí ya sabes cómo se evita.

---
# Parte 8 · Reranking

## Dos etapas: primero amplio, luego fino

Los embeddings comprimen **cada texto por separado** en un vector, y luego se comparan. Es rapidísimo (miles de fragmentos en milisegundos), pero **pierde detalle**: nunca "lee" la pregunta y el fragmento **a la vez**.

El **reranking** añade una segunda etapa:

```
Etapa 1 · RECUPERAR (rápida, amplia)     Etapa 2 · REORDENAR (lenta, fina)
  búsqueda híbrida → 8 candidatos   ──►    un modelo lee pregunta + cada candidato  ──►  los 3 mejores al prompt
```

| | Embeddings (*bi-encoder*) | Reranker (*cross-encoder*) |
| --- | --- | --- |
| Cómo lee | Cada texto **por separado** | Pregunta **y** fragmento **juntos** |
| Velocidad | Muy rápido (se precalcula) | Lento (una pasada por cada candidato) |
| Precisión | Buena | **Mejor** |
| Se usa para | Buscar entre miles | **Reordenar** unas decenas |

🏗️ **Analogía:** la etapa 1 es el **filtro de currículos** que reduce 2 000 candidatos a 8; la etapa 2 es la **entrevista** (cara, lenta, pero solo con 8).

### Qué se usa en producción

Un **cross-encoder especializado** (modelos como `bge-reranker`, o APIs de Cohere, Voyage o Jina). Son más rápidos, baratos y fiables que un LLM general. En esta sesión usamos un **LLM como reranker** porque no exige instalar nada: es el mismo cliente de siempre.

Lo implementamos con **dos versiones de prompt** (versionadas en `prompts/rerank/`, como en la Sesión 3):

- **v1:** "ordena **todos** los fragmentos de más a menos útil".
- **v2:** "elige **solo el mejor**" (una tarea mucho más fácil para un modelo pequeño).

💰 **FYI:** cada pregunta es **1 llamada** con 8 candidatos (≈ 1 300 tokens de entrada). 22 preguntas × 2 versiones ≈ 44 llamadas: **≈ US$ 0.006** con un modelo económico; gratis con `RAG_PROVIDER=local` (tarda 1 o 2 minutos).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.006 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
if PUEDE_GASTAR:
    CANDIDATOS = 8

    def con_rerank(version):
        def buscador(p, k):
            candidatos = buscar_hibrido(indice_base, p["pregunta"], CANDIDATOS)
            resultados, llamada = rerank_llm(p["pregunta"], candidatos, k=k, version=version)
            if llamada:
                gasto.registrar(llamada)
            return resultados
        return buscador

    t0 = time.perf_counter()
    linea("Híbrida, sin rerank (top 3)", evaluar_busqueda(BASE, lambda p, k: buscar_hibrido(indice_base, p["pregunta"], k), 3))
    linea("Híbrida + rerank v1 (ordenar todos)", evaluar_busqueda(BASE, con_rerank("v1"), 3))
    linea("Híbrida + rerank v2 (elegir el mejor)", evaluar_busqueda(BASE, con_rerank("v2"), 3))
    print(f"\n({time.perf_counter() - t0:.0f} s)")
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

## Cómo interpretar esto

- **Un reranker no es magia.** Reordena **solo lo que la etapa 1 trajo**: si el fragmento correcto no está entre los 8 candidatos, nadie lo rescata. Por eso se sube `CANDIDATOS` en la etapa 1 (8 a 50) y se baja `k` en la etapa 2 (3 a 5).
- **Con un modelo pequeño puede empeorar.** Un LLM de 3B parámetros ordena una lista completa **peor** que el orden original; elegir **solo el mejor** (v2) es más fiable, aunque aquí apenas iguala el orden original. Con un modelo de proveedor, o con un cross-encoder, el resultado suele ser claramente mejor. **Tienes que medirlo con tus datos.**
- **Cuesta latencia y dinero**: una llamada extra por pregunta, en el camino crítico de la respuesta.

> 📊 **Referencia del instructor** (modelo local `llama-3.2-3b-instruct`, híbrida con 8 candidatos, 22 preguntas): sin rerank, 18/22 en hit@1 y MRR 0.86; rerank v1, **15/22** y 0.80 (**empeora**); rerank v2, 16 a 17/22 en hit@1 (según la corrida), hit@3 de 20 a **22/22** y MRR 0.86 a 0.89 (no mejora el primer lugar; rescata algún fragmento en el top 3). Los modelos no son del todo deterministas: un resultado puede variar 1 o 2 aciertos entre corridas. No se probó con un modelo de proveedor.

**Reto:** vuelve a ejecutar con `RAG_PROVIDER=anthropic` u `openai` (si tienes llave) y compara. ¿Cambia la conclusión?

---
# Parte 9 · Reescritura de la consulta y multi-consulta

El usuario **no escribe consultas de búsqueda**: escribe como habla. Dos problemas, dos técnicas.

## 9.1 Reescritura: preguntas que no se entienden solas

> **Cliente:** ¿A qué hora atienden por teléfono de lunes a sábado?
> **Asistente:** De 7:00 a 22:00.
> **Cliente:** **¿Y los domingos?**

Si buscas con "¿Y los domingos?", el buscador no sabe de qué hablas. La solución: pedirle a un LLM que **reescriba la última pregunta como una consulta autocontenida** usando el historial: *"¿A qué hora atienden por teléfono los domingos?"*. La plantilla vive en `prompts/reescritura/v1.j2`.

💰 **FYI:** 4 llamadas pequeñas (≈ 300 tokens cada una): **≈ US$ 0.0005**; gratis con `local`.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0005 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
conversacionales = [p for p in preguntas if p["tipo"] == "conversacional"]
reescritas = {}
if PUEDE_GASTAR:
    for p in conversacionales:
        consulta, llamada = reescribir_consulta(p["pregunta"], p["historial"])
        gasto.registrar(llamada)
        reescritas[p["id"]] = consulta
        print(f"{p['id']}  {p['pregunta']!r}\n      → {consulta!r}\n")
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

In [ ]:
if PUEDE_GASTAR and reescritas:
    for nombre, funcion in (("semántica", buscar_vectorial), ("híbrida", buscar_hibrido)):
        sin = evaluar_busqueda(conversacionales, lambda p, k: funcion(indice_base, p["pregunta"], k, p.get("filtro")), 3)
        con = evaluar_busqueda(conversacionales, lambda p, k: funcion(indice_base, reescritas[p["id"]], k, p.get("filtro")), 3)
        print(f"{nombre:<10} SIN reescribir: hit@1 {sin['hit1']}/{sin['n']} · MRR {sin['mrr']:.2f}   |   "
              f"CON reescritura: hit@1 {con['hit1']}/{con['n']} · MRR {con['mrr']:.2f}")

**Lectura:** reescribir ayuda con las preguntas de seguimiento, pero **no es perfecta**: con un modelo pequeño verás consultas que **pierden un dato** del historial (por ejemplo, el nombre del plan) o **lo inventan**. Con solo 4 preguntas es una **ilustración**, no una prueba. Dos reglas prácticas: (1) reescribe **solo cuando hay historial**; (2) **muestra la consulta reescrita** en tus registros: cuando el RAG falle, es lo primero que querrás mirar.

> 📊 **Referencia del instructor** (`llama-3.2-3b-instruct`, 4 preguntas): híbrida, hit@1 de 1/4 a 2/4 y MRR de 0.50 a ≈ 0.60; semántica, de 0/4 a 1/4. En una de las cuatro, el modelo perdió el nombre del plan ("¿Y trae IP fija?" quedó como "¿Trata de IP fija?").

## 9.2 Multi-consulta: preguntar de varias maneras

Otra idea: pedir al LLM **N reformulaciones** de la pregunta, buscar con **todas** y **fusionar** con RRF. Ayuda cuando el usuario usa palabras distintas a las del documento ("multa" vs "penalización").

💰 **FYI:** 9 llamadas pequeñas (una por pregunta semántica): **≈ US$ 0.001**; gratis con `local`.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.001 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
semanticas = [p for p in preguntas if p["tipo"] == "semantica"]
if PUEDE_GASTAR:
    variantes = {}
    for p in semanticas:
        v, llamada = expandir_consulta(p["pregunta"], n=3)
        gasto.registrar(llamada)
        variantes[p["id"]] = v
    print("Ejemplo:", semanticas[1]["pregunta"])
    for v in variantes[semanticas[1]["id"]]:
        print("   ·", v)
    print()
    linea("Semántica (una consulta)", evaluar_busqueda(semanticas, lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k), 3))
    linea("Multi-consulta (original + 3)",
          evaluar_busqueda(semanticas, lambda p, k: buscar_multiconsulta(indice_base, [p["pregunta"]] + variantes[p["id"]], k, modo="vectorial"), 3))
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

**Lectura:** con un modelo pequeño, la multi-consulta **empeora**: las "reformulaciones" **inventan temas** (otro plan, otra garantía) y arrastran fragmentos que no corresponden. Con un modelo fuerte suele ayudar. **Moraleja de las partes 8 y 9: cada paso con LLM añade latencia, dinero y un punto de falla; se queda solo si la medición lo justifica.**

> 📊 **Referencia del instructor** (`llama-3.2-3b-instruct`, 9 preguntas semánticas): una consulta, 8/9 en hit@1 y MRR 0.93; multi-consulta, **5/9** y 0.70.

### HyDE (solo concepto)

*Hypothetical Document Embeddings*: en lugar de vectorizar la pregunta, se le pide al LLM que **escriba una respuesta hipotética** y se vectoriza **esa respuesta** (que se parece más a un documento que una pregunta). Funciona a veces; tiene el mismo problema: si el modelo inventa, la búsqueda se desvía.

---
# Parte 10 · El pipeline completo y la práctica

## Todas las piezas juntas

`responder_avanzado` conecta lo que viste en esta sesión con la generación con citas de la Sesión 5:

```
pregunta ─► [reescribir si hay historial] ─► búsqueda híbrida + FILTRO ─► [rerank] ─► prompt con citas ─► LLM ─► verificar citas
```

Los pasos entre corchetes son **opcionales**: activa solo los que **mediste que ayudan**. Devuelve la respuesta y un diccionario con lo que pasó en cada paso (para **depurar**).

💰 **FYI:** 3 preguntas; con todos los pasos, ≈ 3 llamadas cada una: **≈ US$ 0.003**; gratis con `local`.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.003 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
casos = [
    dict(pregunta="¿Qué indica la alarma OPT-1107?"),
    dict(pregunta="¿Y los domingos?", historial=conversacionales[0]["historial"], reescribir=True),
    dict(pregunta="¿Cuánto dura el contrato?", filtro={"publico": "negocio"}),
]
if PUEDE_GASTAR:
    for caso in casos:
        respuesta, pasos = responder_avanzado(indice_base, **caso)
        for nombre, llamada in pasos["llamadas"]:
            gasto.registrar(llamada)
        print(f"━━ {caso['pregunta']}")
        if pasos["consulta"] != caso["pregunta"]:
            print(f"   consulta reescrita → {pasos['consulta']}")
        print(f"   respuesta: {respuesta.texto}")
        print(f"   recuperó:  {[f.metadatos['fuente'] + ' · ' + f.metadatos['seccion'] for f in respuesta.fragmentos]}")
        print(f"   citó:      {sorted(respuesta.citas) or 'nada (el modelo no citó ningún fragmento)'} · citas inválidas: {sorted(respuesta.citas_invalidas) or 'ninguna'}\n")
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

Fíjate en dos cosas: la **consulta reescrita** aparece en el diccionario de pasos (así se depura), y el modelo pequeño puede **responder bien y no citar**: en la Sesión 5 viste que citar es difícil para un modelo de 3B. Las respuestas sin cita **no son verificables**; con un modelo de proveedor suelen citar mejor.

## La práctica: Laboratorio de chunking

La práctica de la sesión es [`practica/laboratorio_chunking.py`](practica/laboratorio_chunking.py): un programa único y comentado que **prueba combinaciones de estrategia × tamaño × tipo de búsqueda** con tus documentos y tus preguntas, y te dice **cuál recupera mejor y a qué costo de contexto**. No llama al LLM: es gratis.

```bash
# Desde la carpeta s06-rag-intermedio/
uv run python practica/laboratorio_chunking.py
uv run python practica/laboratorio_chunking.py --modo hibrido --detalle
uv run python practica/laboratorio_chunking.py --estrategias padre-hijo,seccion,oraciones --tamanos 35,70
uv run python practica/laboratorio_chunking.py --tipos lexica
uv run python practica/laboratorio_chunking.py --documentos mi_carpeta/ --preguntas mis_preguntas.json
```

### Retos
1. **Tu propio corpus:** añade un documento a `datos/` (un reglamento, un manual, una página web guardada) y escribe **8 preguntas** con su `clave` en un JSON nuevo. ¿Gana la misma estrategia?
2. **El tamaño ideal:** prueba `--tamanos 20,40,70,120,200`. ¿Dónde empieza a empeorar? ¿Por qué?
3. **Una pregunta difícil:** escribe una pregunta que **ninguna** configuración responda bien y explica por qué (¿tabla aplanada? ¿término que el documento no usa?).
4. **Prompt de reescritura v2:** crea `prompts/reescritura/v2.j2`, anótalo en `prompts/CHANGELOG.md` y compáralo con v1 en las cuatro preguntas conversacionales.
5. **Detectar escaneos:** escribe una función que recorra una carpeta y liste los PDFs con páginas sin texto.

---
# Parte 11 · Cierre

## Lo que aprendiste
1. La **ingesta** decide la calidad de todo lo demás: extrae texto **limpio y estructurado**; un escaneo sin OCR **no existe** para tu RAG.
2. El **chunking** equilibra precisión al buscar y contexto al responder; se elige **midiendo**, mirando también el **costo del contexto**.
3. **Padre-hijo:** buscar con fragmentos pequeños y entregar el grande.
4. La búsqueda **semántica** entiende paráfrasis; **BM25** no se confunde con códigos. La **híbrida con RRF** combina ambas, y los **pesos** importan.
5. Los **metadatos y filtros** resuelven reglas de negocio y de **seguridad** antes de que los datos lleguen al modelo.
6. El **reranking** y la **reescritura de la consulta** pueden ayudar, pero cada uno añade un LLM: **se justifican con mediciones**, no con moda.
7. Con pocas preguntas, las diferencias pequeñas son **ruido**.

## ✅ Autoevaluación
1. ¿Qué dos presiones opuestas hay al elegir el tamaño de un fragmento?
2. ¿Por qué la estrategia `seccion` "gana" en MRR y por qué eso puede ser engañoso?
3. Da un ejemplo de pregunta donde BM25 gane a la semántica y otro donde pierda.
4. ¿Por qué RRF usa la **posición** y no el puntaje?
5. ¿Por qué filtrar por permisos en la búsqueda y no en el prompt?
6. ¿Qué diferencia hay entre un *bi-encoder* y un *cross-encoder*? ¿Cuál usarías para reordenar 8 candidatos?
7. ¿Cuándo **no** reescribirías la consulta?
8. ¿Qué es el OCR y cómo te darías cuenta de que a tu RAG le faltan documentos escaneados?

## 💰 Resumen de gasto de esta sesión

In [ ]:
gasto.resumen()

## 📖 Glosario

| Término | Definición |
| --- | --- |
| **Ingesta** | Extraer el texto de los documentos, fragmentarlo, vectorizarlo y guardarlo |
| **OCR** | Convertir una imagen de texto (un escaneo) en texto |
| **Chunking** | Dividir un documento en fragmentos para buscar y para dar contexto |
| **Solape (*overlap*)** | Repetir el final de un fragmento al inicio del siguiente |
| **Fragmentado recursivo** | Cortar por el separador más grande que quepa (párrafo, línea, oración, palabra) |
| **Fragmentado semántico** | Cortar donde cambia el tema según la similitud entre oraciones |
| **Padre-hijo** | Buscar con fragmentos pequeños (hijos) y entregar el fragmento grande (padre) |
| **BM25** | Algoritmo clásico de búsqueda por palabras (IDF + frecuencia + longitud) |
| **IDF** | Qué tan rara es una palabra en el corpus: las raras pesan más |
| **Búsqueda híbrida** | Combinar búsqueda por palabras y búsqueda semántica |
| **RRF** | *Reciprocal Rank Fusion*: combina listas ordenadas usando solo la posición |
| **Metadatos** | Datos sobre cada fragmento (fuente, sección, audiencia) |
| **Filtro** | Restricción por metadatos aplicada **dentro** de la búsqueda |
| **Reranking** | Reordenar los candidatos con un modelo que lee pregunta y fragmento juntos |
| **Cross-encoder** | Modelo que puntúa pregunta y fragmento **juntos**; más preciso y lento que un bi-encoder |
| **Reescritura de consulta** | Convertir una pregunta de seguimiento en una consulta autocontenida |
| **Multi-consulta** | Buscar con varias reformulaciones y fusionar los resultados |
| **HyDE** | Vectorizar una respuesta hipotética en lugar de la pregunta |
| **hit@k** | La respuesta está entre los k primeros resultados |
| **MRR** | Promedio de 1/posición del primer acierto |

## ➡️ Siguiente sesión
Ya sabes **recuperar bien**. El siguiente paso es **medir si el RAG responde bien y cuándo inventa**: evaluación de la generación, *groundedness* (fundamentación) y alucinaciones.